# 06 - Integrated Pipeline: Case-Crossover → Shrinkage → Recommendations

This notebook validates the end-to-end SafePlate pipeline by connecting:

1. **Case-crossover output** from notebook 04 (`ingredient_stats_all.csv`)
2. **Empirical Bayes shrinkage** via `GroupAwareTriggerDetector`
3. **Recommendation engine** for safe-substitute queries

The integration layer (`SafePlatePipeline`) is defined in `backend/safeplate/integrated_pipeline.py`.

**Date:** February 2026  
**Author:** Adar Palaoglu

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Add backend to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "backend"))

from safeplate.embedding_loader import EmbeddingLoader
from safeplate.ingredient_groups import IngredientGroupManager
from safeplate.integrated_pipeline import SafePlatePipeline, PipelineConfig

print(f"Project root: {project_root}")
print("Imports OK")

## Load Data and Initialise Components

In [ ]:
# Initialise embedding loader
embedding_path = (
    project_root / "data" / "processed" / "embeddings"
    / "ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w10_wl80_r80_e10.txt"
)
assert embedding_path.exists(), f"Embedding file not found: {embedding_path}"

loader = EmbeddingLoader(str(embedding_path))
print(f"Embedding loader: {loader.num_ingredients} ingredients, {loader.embedding_dim} dimensions")

In [ ]:
# Initialise group manager
groups_path = project_root / "data" / "processed" / "ingredient_groups.csv"
assert groups_path.exists(), f"Groups file not found: {groups_path}"

gm = IngredientGroupManager(str(groups_path))
gm.load()
print(f"Group manager: {gm.num_ingredients} ingredients, {gm.num_groups} groups")

In [ ]:
# Load case-crossover output from notebook 04
stats_path = project_root / "data" / "processed" / "trigger_detection" / "ingredient_stats_all.csv"

if not stats_path.exists():
    print("=" * 70)
    print("ERROR: ingredient_stats_all.csv not found.")
    print(f"Expected path: {stats_path}")
    print()
    print("You must run notebook 04_trigger_detection_recommended.ipynb first")
    print("to produce the case-crossover output.")
    print("=" * 70)
    raise FileNotFoundError(f"Run notebook 04 first: {stats_path}")

ingredient_stats = pd.read_csv(stats_path)
print(f"Loaded ingredient_stats: {len(ingredient_stats)} rows")
print(f"Columns: {list(ingredient_stats.columns)}")
display(ingredient_stats.head(10))

## Initialise the Integrated Pipeline

In [ ]:
# Create pipeline with default config
config = PipelineConfig(
    trigger_or_threshold=1.0,
    trigger_min_confidence="MEDIUM",
    propagate_to_untested=True,
    recommendation_top_n=10,
)

pipeline = SafePlatePipeline(
    embedding_loader=loader,
    group_manager=gm,
    config=config,
)

print("Pipeline initialised")
print(f"  trigger_or_threshold   = {config.trigger_or_threshold}")
print(f"  trigger_min_confidence = {config.trigger_min_confidence}")
print(f"  propagate_to_untested  = {config.propagate_to_untested}")
print(f"  recommendation_top_n   = {config.recommendation_top_n}")

## Run Shrinkage Model on Case-Crossover Results

In [ ]:
# Run the shrinkage model
shrinkage_result = pipeline.detect_triggers_from_stats(ingredient_stats)

print("=" * 70)
print("SHRINKAGE MODEL COMPLETE")
print("=" * 70)
print(f"Total ingredients in output: {len(shrinkage_result.ingredients)}")
print(f"Groups processed:            {len(shrinkage_result.groups)}")

## Pipeline Summary Statistics

In [ ]:
summary = pipeline.get_pipeline_summary(shrinkage_result)

print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)
for key, value in summary.items():
    if isinstance(value, list):
        if len(value) <= 10:
            print(f"  {key}: {value}")
        else:
            print(f"  {key}: [{', '.join(value[:10])}, ... +{len(value)-10} more]")
    else:
        print(f"  {key}: {value}")

## Test Substitution Recommendations

In [ ]:
# Find safe substitutes for a test ingredient
from safeplate.recommendation_engine import IngredientNotFoundError

test_queries = ["milk", "tomato", "cheese"]

trigger_list = pipeline.get_trigger_ingredients(shrinkage_result)
print("=" * 70)
print("SUBSTITUTION RECOMMENDATIONS")
print("=" * 70)
print(f"\nActive triggers being excluded ({len(trigger_list)}):")
for t in trigger_list[:20]:
    print(f"  - {t}")
if len(trigger_list) > 20:
    print(f"  ... and {len(trigger_list) - 20} more")

for query in test_queries:
    print(f"\n--- Substitutes for '{query}' ---")
    try:
        subs = pipeline.get_safe_substitutes(query, shrinkage_result, top_n=5)
        for i, (ing, score) in enumerate(subs, 1):
            print(f"  {i}. {ing:<30} (similarity: {score:.4f})")
    except IngredientNotFoundError as e:
        print(f"  [NOT FOUND] {e}")

## Validate Edge Cases

In [ ]:
print("=" * 70)
print("EDGE CASE VALIDATION")
print("=" * 70)

all_results = shrinkage_result.ingredients

# 1. Protected safe ingredients
protected = [r for r in all_results if r.protected]
print(f"\n1. Protected safe ingredients: {len(protected)}")
for r in protected:
    print(f"   - {r.ingredient}: U={r.U}, V={r.V}, raw_OR={r.raw_OR:.3f}, "
          f"group={r.group}")

# 2. Untested ingredients (inherited from group mean)
untested = [r for r in all_results if r.U == 0 and r.V == 0]
print(f"\n2. Untested ingredients (group_mean inheritors): {len(untested)}")
if untested:
    for r in untested[:10]:
        print(f"   - {r.ingredient}: final_OR={r.final_OR:.3f}, "
              f"group={r.group}, confidence={r.confidence}")
    if len(untested) > 10:
        print(f"   ... and {len(untested) - 10} more")

# 3. Groups with high tau_sq (heterogeneous)
group_results = shrinkage_result.groups
hetero_groups = [g for g in group_results if g.tau_sq > 0]
print(f"\n3. Heterogeneous groups (tau_sq > 0): {len(hetero_groups)}")
for g in sorted(hetero_groups, key=lambda x: x.tau_sq, reverse=True)[:5]:
    print(f"   - {g.group}: tau_sq={g.tau_sq:.4f}, n_tested={g.n_tested}, "
          f"n_in_pool={g.n_in_pool}")

# 4. Groups with tau_sq = 0 (homogeneous / maximum pooling)
homo_groups = [g for g in group_results if g.tau_sq == 0]
print(f"\n4. Homogeneous groups (tau_sq = 0): {len(homo_groups)}")
for g in homo_groups[:5]:
    print(f"   - {g.group}: mu_g={g.mu_g:.4f}, n_tested={g.n_tested}, "
          f"n_in_pool={g.n_in_pool}")

## Comparison: Raw OR vs Shrunk OR

In [ ]:
# Build comparison table for tested ingredients only
tested_results = [r for r in all_results if r.U + r.V > 0]

comparison_data = []
for r in tested_results:
    comparison_data.append({
        "ingredient": r.ingredient,
        "U": r.U,
        "V": r.V,
        "raw_OR": round(r.raw_OR, 3),
        "final_OR": round(r.final_OR, 3),
        "weight": round(r.weight, 3),
        "group": r.group or "(none)",
        "confidence": r.confidence,
        "evidence_source": r.evidence_source,
        "protected": r.protected,
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values("final_OR", ascending=False)

print("=" * 70)
print("RAW OR vs SHRUNK OR (tested ingredients)")
print("=" * 70)
display(comparison_df)

## Summary

### What was validated

1. **Pipeline initialisation** -- all three components (embeddings, groups, shrinkage) load and connect.
2. **Shrinkage model** -- case-crossover output flows through the D&L estimator correctly.
3. **Trigger detection** -- ingredients are classified with appropriate confidence labels.
4. **Substitution recommendations** -- the recommendation engine excludes triggers from results.
5. **Edge cases** -- protected-safe, untested inheritors, heterogeneous/homogeneous groups.

### Next steps

- Build synthetic test lab with known ground-truth triggers for quantitative evaluation.
- Wire into Flask web UI for the dissertation demonstration.
- Compare shrinkage approach against baselines (no grouping, fixed-effect, naive propagation).